**Data Patterns and Representations: Final Project** <br>
**2/24/2026** <br>
**Group 9:** Sadia Rahman | Brooke Rice

# Representation-Specific Analysis

## Data Representation and Preprocessing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
# load data
repatriations = pd.read_excel("KHSM Repatriations fy25m11.xlsx", sheet_name = None)
enc_2020_2023 = pd.read_excel("nationwide-encounters-fy20-fy23-aor.xlsx")
enc_2023_2026 = pd.read_excel("nationwide-encounters-fy23-fy26-jan-aor.xlsx")
policy_monthly = pd.read_csv("policy_markers_monthly_2015_2025.csv")
#policy_events = pd.read_csv("policy_timeline_events_2015_2025.csv")

The raw datasets were loaded from multiple sources, including DHS repatriation data, CBP encounter data, and a custom policy marker dataset. The repatriation data came from a multisheet Excel file, while the encounter data were split across two Excel files covering different fiscal year ranges. The policy dataset was loaded from a CSV file containing monthly policy indicators. All datasets were imported into pandas DataFrames so they could be cleaned and processed consistently across sources.

### Clean DHS Monthly Repatriations

In [3]:
# extract the relevant sheet
dhs_monthly = repatriations["Monthly Repatriations"].copy()

# standardize column names
dhs_monthly.columns = (dhs_monthly.columns.astype(str)
                       .str.replace("\n", " ", regex = False)
                       .str.strip()
                       )

# rename columns for consistency
dhs_monthly = dhs_monthly.rename(columns = {"Fiscal Year": "year",
                                            "Month": "month_label",
                                            "Quantity": "repatriations"
                                            })

dhs_monthly["year"] = pd.to_numeric(dhs_monthly["year"], errors = "coerce") # convert year to numeric
dhs_monthly["month_num"] = dhs_monthly["month_label"].str[:2].astype(int) # extract fiscal month index
dhs_monthly["date"] = pd.to_datetime(dict(year = dhs_monthly["year"], month = dhs_monthly["month_num"], day = 1)) # build proper calendar date
dhs_monthly = dhs_monthly[["date", "year", "month_num", "repatriations"]].dropna() # keep only relevant columns

print("DHS Monthly Shape:", dhs_monthly.shape)
print("\nSample rows:")
print(dhs_monthly.head())
print("\nMissing values per column:")
print(dhs_monthly.isna().sum())
print("\nDuplicate dates:", dhs_monthly["date"].duplicated().sum())
print("\nDate range:", dhs_monthly["date"].min(), "to", dhs_monthly["date"].max())
print("\nRepatriations summary stats:")
print(dhs_monthly["repatriations"].describe())

# check for missing months
expected_range = pd.date_range(start = dhs_monthly["date"].min(),
                               end = dhs_monthly["date"].max(),
                               freq = "MS"
                               )

missing_months = expected_range.difference(dhs_monthly["date"])
print("\nMissing months in sequence:", len(missing_months))
print(missing_months[:5])

DHS Monthly Shape: (182, 4)

Sample rows:
        date  year  month_num  repatriations
0 2010-01-01  2010          1          79200
1 2010-02-01  2010          2          69680
2 2010-03-01  2010          3          63560
3 2010-04-01  2010          4          69200
4 2010-05-01  2010          5          75750

Missing values per column:
date             0
year             0
month_num        0
repatriations    0
dtype: int64

Duplicate dates: 0

Date range: 2010-01-01 00:00:00 to 2025-02-01 00:00:00

Repatriations summary stats:
count       182.000000
mean      61840.714286
std       29037.522287
min       27080.000000
25%       40067.500000
50%       51780.000000
75%       71817.500000
max      147080.000000
Name: repatriations, dtype: float64

Missing months in sequence: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')


The DHS repatriation dataset was cleaned and transformed into a standardized monthly time series by extracting the “Monthly Repatriations” sheet from the source workbook. Column names were first normalized to remove formatting inconsistencies, and key variables were renamed for clarity. The fiscal year and month label fields were converted into numeric representations, and a unified monthly date variable was constructed to enable alignment with other datasets. Validation checks confirmed that the dataset contained no missing values, no duplicate dates, and complete monthly coverage across the full time range. A continuity check verified that there were no gaps in the monthly sequence, ensuring the dataset is consistently structured at one observation per month. Summary statistics further confirmed reasonable variation in repatriation counts over time. Overall, the DHS dataset was successfully cleaned and validated, providing a reliable foundation for integration with encounter and policy data.

### Combine and Clean CBP encounters

In [4]:
# remove FY2023 overlap
enc_2020_2023 = enc_2020_2023[enc_2020_2023["Fiscal Year"] != 2023]

# fix Fiscal Year column in second dataset
enc_2023_2026["Fiscal Year"] = enc_2023_2026["Fiscal Year"].apply(lambda x: 2026 if "2026" in str(x) else int(x))

# combine datasets
encounters = pd.concat([enc_2020_2023, enc_2023_2026], ignore_index = True)

# clean column names
encounters.columns = encounters.columns.str.strip()

# rename columns to consistent format
encounters = encounters.rename(columns = {"Fiscal Year": "year",
                                          "Month (abbv)": "month_abbr",
                                          "Encounter Count": "encounters",
                                          "Title of Authority": "title_authority"
                                         })

encounters["month_abbr"] = encounters["month_abbr"].astype(str).str.upper().str[:3] # ensure month abbreviations are clean

# map month abbreviations
month_map = {"OCT": 10, "NOV": 11, "DEC": 12, 
             "JAN": 1, "FEB": 2, "MAR": 3,
             "APR": 4, "MAY": 5, "JUN": 6,
             "JUL": 7, "AUG": 8, "SEP": 9
            }

encounters["month_num"] = encounters["month_abbr"].map(month_map)
print("Missing month_num values:", encounters["month_num"].isna().sum()) # check for mapping issues

# convert fiscal to calendar year
encounters["calendar_year"] = encounters["year"]
encounters.loc[encounters["month_num"].isin([10, 11, 12]), "calendar_year"] = encounters["year"] - 1

# create proper monthly date
encounters["date"] = pd.to_datetime(dict(year = encounters["calendar_year"], month = encounters["month_num"], day = 1))

# aggregate to monthly level
monthly_encounters = (encounters
                      .groupby("date", as_index = False)["encounters"]
                      .sum()
                     )

# validation
print("\nMonthly encounters shape:", monthly_encounters.shape)
print("Duplicate dates:", monthly_encounters["date"].duplicated().sum())
print("Missing values:\n", monthly_encounters.isna().sum())
print("\nSample rows:")
print(monthly_encounters.head())
print("\nDate range:", monthly_encounters["date"].min(), "to", monthly_encounters["date"].max())
print("\nSummary statistics:")
print(monthly_encounters["encounters"].describe())

# check for missing months
expected_range = pd.date_range(start = monthly_encounters["date"].min(),
                               end = monthly_encounters["date"].max(),
                               freq = "MS"
                               )

missing_months = expected_range.difference(monthly_encounters["date"])
print("\nMissing months in sequence:", len(missing_months))
print(missing_months[:5])

Missing month_num values: 0

Monthly encounters shape: (76, 2)
Duplicate dates: 0
Missing values:
 date          0
encounters    0
dtype: int64

Sample rows:
        date  encounters
0 2019-10-01       61159
1 2019-11-01       57524
2 2019-12-01       56186
3 2020-01-01       52254
4 2020-02-01       54884

Date range: 2019-10-01 00:00:00 to 2026-01-01 00:00:00

Summary statistics:
count        76.000000
mean     161714.710526
std       99942.556022
min       24589.000000
25%       55860.500000
50%      188857.000000
75%      246759.500000
max      370883.000000
Name: encounters, dtype: float64

Missing months in sequence: 0
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')


The CBP encounter data came from two files that overlapped in fiscal year 2023, so that year was removed from the earlier dataset before combining them. After that, the files were stacked together into one dataset for cleaning. Column names were standardized, and the month abbreviations were converted into numeric values so a proper date could be created. Since the data is recorded in fiscal years, October through December were adjusted to the previous calendar year to get the correct timeline. This allowed everything to line up with the DHS data. Because the raw data has multiple rows per month, it was grouped and summed to get one total per month. The final dataset has 76 monthly observations from October 2019 through January 2026, with no missing values, no duplicate dates, and no gaps in the timeline. Monthly encounters average around 161,715, with values ranging from about 24,589 to 370,883, showing a lot of variation over time.

### Clean Policy Marker and Policy Timeline

In [5]:
# convert month column to datetime
policy_monthly["date"] = pd.to_datetime(policy_monthly["month"], errors = "coerce")

# clean column names
policy_monthly.columns = policy_monthly.columns.str.strip()

# rename for consistency
policy_monthly = policy_monthly.rename(columns = {"admin": "administration"})

# keep relevant columns
policy_monthly = policy_monthly[["date",
                                 "administration",
                                 "phe_covid",
                                 "title42",
                                 "border_national_emergency",
                                 "mpp_active",
                                 "clp_rule_active",
                                 "mayorkas_priorities_active",
                                 "title42_uac_exempt"
                                ]]

# validation
print("Policy dataset shape:", policy_monthly.shape)
print("\nSample rows:")
print(policy_monthly.head())
print("\nMissing values per column:")
print(policy_monthly.isna().sum())
print("\nDuplicate dates:", policy_monthly["date"].duplicated().sum())
print("\nDate range:", policy_monthly["date"].min(), "to", policy_monthly["date"].max())

# check for missing months
expected_range = pd.date_range(start = policy_monthly["date"].min(),
                               end = policy_monthly["date"].max(),
                               freq = "MS"
                              )

missing_months = expected_range.difference(policy_monthly["date"])
print("\nMissing months in sequence:", len(missing_months))
print(missing_months[:5])

Policy dataset shape: (132, 9)

Sample rows:
        date administration  phe_covid  title42  border_national_emergency  \
0 2015-01-01          Obama          0        0                          0   
1 2015-02-01          Obama          0        0                          0   
2 2015-03-01          Obama          0        0                          0   
3 2015-04-01          Obama          0        0                          0   
4 2015-05-01          Obama          0        0                          0   

   mpp_active  clp_rule_active  mayorkas_priorities_active  title42_uac_exempt  
0           0                0                           0                   0  
1           0                0                           0                   0  
2           0                0                           0                   0  
3           0                0                           0                   0  
4           0                0                           0                   0  


The policy marker dataset was cleaned by converting the raw month column into a proper datetime variable and standardizing the column names. The administration column was renamed for clarity, and only the relevant policy indicator variables were kept to keep the dataset focused and consistent with the analysis. The cleaned dataset contains 132 monthly observations from January 2015 through December 2025. There are no missing values, no duplicate dates, and no gaps in the monthly sequence, confirming that the dataset is complete and consistently structured at one observation per month.

### Merge all monthly sources

In [6]:
df = (dhs_monthly
    .merge(monthly_encounters, on = "date", how = "inner") # core alignment
    .merge(policy_monthly, on = "date", how = "left") # keep all DHS/CBP months
     )

# fill policy related missing values (pre policy periods)
policy_cols = ["phe_covid",
               "title42",
               "border_national_emergency",
               "mpp_active",
               "clp_rule_active",
               "mayorkas_priorities_active",
               "title42_uac_exempt"]

df[policy_cols] = df[policy_cols].fillna(0)

# validation
print("Merged dataset shape:", df.shape)
print("\nSample merged rows:")
print(df.head())
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate dates:", df["date"].duplicated().sum())
print("\nDate range:", df["date"].min(), "to", df["date"].max())

# checking for missing months
expected_range = pd.date_range(start = df["date"].min(),
                               end = df["date"].max(),
                               freq = "MS"
                              )

missing_months = expected_range.difference(df["date"])
print("\nMissing months in merged dataset:", len(missing_months))
print(missing_months[:5])

Merged dataset shape: (65, 13)

Sample merged rows:
        date  year  month_num  repatriations  encounters administration  \
0 2019-10-01  2019         10          43580       61159          Trump   
1 2019-11-01  2019         11          43190       57524          Trump   
2 2019-12-01  2019         12          41990       56186          Trump   
3 2020-01-01  2020          1          51360       52254          Trump   
4 2020-02-01  2020          2          46120       54884          Trump   

   phe_covid  title42  border_national_emergency  mpp_active  clp_rule_active  \
0          0        0                          1           1                0   
1          0        0                          1           1                0   
2          0        0                          1           1                0   
3          1        0                          1           1                0   
4          1        0                          1           1                0   

   mayorka

After cleaning each dataset, the DHS repatriation data, CBP encounter data, and policy marker dataset were merged into a single table using the shared monthly date field. An inner join was used between the DHS and CBP datasets to keep only months present in both core sources, while the policy data was merged in using a left join so that all months were retained even if no policy indicators were active. The final merged dataset contains 65 monthly observations spanning from October 2019 to February 2025, with one row per month. Validation checks confirmed that there are no missing values across any variables, no duplicate dates, and no gaps in the monthly sequence, showing that the datasets were aligned correctly. Policy indicator variables were filled with zero where needed to represent periods where policies were not active rather than missing data. Overall, the merge resulted in a clean, consistent, and fully aligned dataset that is ready for feature engineering and further analysis.

### Feature Engineering

In [7]:
df["repat_to_enc_ratio"] = df["repatriations"] / df["encounters"] # core ratio
df["title42_share"] = df["title42"] # policy feature

# temporal features
df["lag_1_ratio"] = df["repat_to_enc_ratio"].shift(1)
df["rolling_3mo_ratio"] = df["repat_to_enc_ratio"].rolling(3).mean()

# administration feature
print("\nColumns BEFORE encoding:")
print(df.columns)

if "administration" in df.columns:
    df = pd.get_dummies(df, columns=["administration"], drop_first=True)
else:
    print("skipping encoding of admin (either missing or already encoded")

print("\nColumns AFTER encoding:")
print(df.columns)

# validation
print("\nFeature engineered dataset shape:", df.shape)
print("\nSample feature rows:")
print(df[["date",
          "repatriations",
          "encounters",
          "repat_to_enc_ratio",
          "title42_share",
          "lag_1_ratio",
          "rolling_3mo_ratio"
          ]].head(10))
print("\nMissing values in engineered features:")
print(df[["repat_to_enc_ratio",
          "title42_share",
          "lag_1_ratio",
          "rolling_3mo_ratio"
          ]].isna().sum())
print("\nRatio summary:")
print(df["repat_to_enc_ratio"].describe())
print("\nMonths with zero encounters:", (df["encounters"] == 0).sum())
print("Infinite values in ratio:", np.isinf(df["repat_to_enc_ratio"]).sum())

# final dataset preview
print("Final dataset shape:", df.shape)
display(df.head(10))


Columns BEFORE encoding:
Index(['date', 'year', 'month_num', 'repatriations', 'encounters',
       'administration', 'phe_covid', 'title42', 'border_national_emergency',
       'mpp_active', 'clp_rule_active', 'mayorkas_priorities_active',
       'title42_uac_exempt', 'repat_to_enc_ratio', 'title42_share',
       'lag_1_ratio', 'rolling_3mo_ratio'],
      dtype='object')

Columns AFTER encoding:
Index(['date', 'year', 'month_num', 'repatriations', 'encounters', 'phe_covid',
       'title42', 'border_national_emergency', 'mpp_active', 'clp_rule_active',
       'mayorkas_priorities_active', 'title42_uac_exempt',
       'repat_to_enc_ratio', 'title42_share', 'lag_1_ratio',
       'rolling_3mo_ratio', 'administration_Trump'],
      dtype='object')

Feature engineered dataset shape: (65, 17)

Sample feature rows:
        date  repatriations  encounters  repat_to_enc_ratio  title42_share  \
0 2019-10-01          43580       61159            0.712569              0   
1 2019-11-01          4

,date,year,month_num,repatriations,encounters,phe_covid,title42,border_national_emergency,mpp_active,clp_rule_active,mayorkas_priorities_active,title42_uac_exempt,repat_to_enc_ratio,title42_share,lag_1_ratio,rolling_3mo_ratio,administration_Trump
0,2019-10-01,2019,10,43580,61159,0,0,1,1,0,0,0,0.712569,0,NaN,NaN,True
1,2019-11-01,2019,11,43190,57524,0,0,1,1,0,0,0,0.750817,0,0.712569,NaN,True
2,2019-12-01,2019,12,41990,56186,0,0,1,1,0,0,0,0.747339,0,0.750817,0.736908,True
3,2020-01-01,2020,1,51360,52254,1,0,1,1,0,0,0,0.982891,0,0.747339,0.827016,True
4,2020-02-01,2020,2,46120,54884,1,0,1,1,0,0,0,0.840318,0,0.982891,0.856849,True
5,2020-03-01,2020,3,44920,51869,1,1,1,1,0,0,0,0.866028,1,0.840318,0.896412,True
6,2020-04-01,2020,4,47060,29743,1,1,1,1,0,0,0,1.582221,1,0.866028,1.096189,True
7,2020-05-01,2020,5,48730,39877,1,1,1,1,0,0,0,1.222008,1,1.582221,1.223419,True
8,2020-06-01,2020,6,50560,50086,1,1,1,1,0,0,0,1.009464,1,1.222008,1.271231,True
9,2020-07-01,2020,7,36960,53672,1,1,1,1,0,0,0,0.688627,1,1.009464,0.973366,True


At the end of preprocessing and feature engineering, the final dataset contains 65 monthly observations with all variables aligned and cleaned. It includes repatriation counts, encounter totals, policy indicators, and engineered features like the repatriation to encounter ratio, lagged values, and a three month rolling average. The administration variable was also encoded to prepare the data for modeling. A final check confirmed there are no missing values in key fields and that the dataset is consistently structured across all columns. Overall, the data is complete and ready for analysis or modeling.

In [8]:
# check raw DHS sheet BEFORE any cleaning
raw_dhs = repatriations["Monthly Repatriations"]

print(raw_dhs.columns)
print(raw_dhs.head(15))

Index(['Fiscal\nYear', 'Month', 'Quantity'], dtype='object')
    Fiscal\nYear         Month  Quantity
0           2010    01 October     79200
1           2010   02 November     69680
2           2010   03 December     63560
3           2010    04 January     69200
4           2010   05 February     75750
5           2010      06 March    102640
6           2010      07 April     96020
7           2010        08 May     80900
8           2010       09 June     76950
9           2010       10 July     69280
10          2010     11 August     75710
11          2010  12 September     72290
12          2011    01 October     56010
13          2011   02 November     53430
14          2011   03 December     52550


In [10]:
final_df = df.copy()
final_df.to_csv("final_model_dataset.csv", index = False)
print("Saved final dataset:", final_df.shape)

Saved final dataset: (65, 17)
